# Hospital Patient Survival Forecasting



## Section 1 — Environment Setup & Imports

In [ ]:
# Install kaggle if not already present
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kaggle', '-q'], check=True)
print("kaggle package ready.")

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import os, zipfile, warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})

# Preprocessing & models
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# Validation
from sklearn.model_selection import train_test_split, GroupKFold

# Metrics
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix
)

print("All imports successful.")

## Section 2 — Download Dataset via Kaggle API

### Source: https://www.kaggle.com/c/widsdatathon2020

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your kaggle.json
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('kaggle.json configured.')

# ── All environments: check kaggle.json is in place ──────────────────────────
kaggle_cfg = os.path.expanduser('~/.kaggle/kaggle.json')
if os.path.exists(kaggle_cfg):
    os.chmod(kaggle_cfg, 0o600)
    print(f"kaggle.json found at: {kaggle_cfg}")
else:
    print("kaggle.json NOT found. Please follow the prerequisites above.")

In [ ]:
# Download the WiDS 2020 training data using the Kaggle API
DOWNLOAD_DIR   = './wids_data'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("Downloading WiDS Datathon 2020 dataset...")
os.system(
    f'kaggle competitions download -c widsdatathon2020 '
    f'-f training_v2.csv -p {DOWNLOAD_DIR}'
)

# Handle zip if downloaded as archive
zip_path  = os.path.join(DOWNLOAD_DIR, 'training_v2.csv.zip')
csv_path  = os.path.join(DOWNLOAD_DIR, 'training_v2.csv')

if os.path.exists(zip_path) and not os.path.exists(csv_path):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DOWNLOAD_DIR)
    print('Extracted from zip.')

if os.path.exists(csv_path):
    print(f'Dataset ready: {csv_path}')
else:
    print('File not found — check your Kaggle credentials and competition acceptance.')

## Section 3 — Load & Explore the Data

In [ ]:
# Load into DataFrame
patient_records = pd.read_csv(csv_path)
print(f"Shape: {patient_records.shape}")
print(f"Rows: {patient_records.shape[0]:,}  |  Columns: {patient_records.shape[1]}")

In [ ]:
patient_records.head(5)

In [ ]:
# Encounter-level stats
total_encounters = patient_records['encounter_id'].nunique()
print(f"Unique encounter IDs : {total_encounters:,}")
print(f"Unique patient IDs   : {patient_records['patient_id'].nunique():,}")
print(f"Unique hospital IDs  : {patient_records['hospital_id'].nunique():,}")

In [ ]:
# Target variable distribution
outcome_counts = patient_records['hospital_death'].value_counts()
pct_survived   = outcome_counts[0] / len(patient_records) * 100
pct_fatal      = outcome_counts[1] / len(patient_records) * 100

print(f"Class 0 (survived): {outcome_counts[0]:,}  → {pct_survived:.1f}%")
print(f"Class 1 (died)    : {outcome_counts[1]:,}  → {pct_fatal:.1f}%")

# Simple donut chart
fig, ax = plt.subplots(figsize=(5, 5))
wedges, texts, pcts = ax.pie(
    [outcome_counts[0], outcome_counts[1]],
    labels=['Survived', 'Died'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['#4C9BE8', '#E85757'],
    wedgeprops=dict(width=0.55)
)
ax.set_title("Target Variable Distribution", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

##  Section 4 — Missing Value Summary

In [ ]:
def get_null_report(dataframe):
    """Builds a summary table of null counts and percentages."""
    null_counts  = dataframe.isnull().sum().sort_values(ascending=False)
    null_pcts    = (dataframe.isnull().mean() * 100).sort_values(ascending=False)
    report       = pd.concat([null_counts, null_pcts], axis=1,
                             keys=['Null Count', 'Null %'])
    return report[report['Null Count'] > 0]

null_report = get_null_report(patient_records)
print(f"Columns with nulls: {len(null_report)}")
null_report.head(25)

In [ ]:
# Visual: top 25 most-missing features
top25_null = null_report.head(25)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top25_null.index[::-1], top25_null['Null %'][::-1],
               color='salmon', edgecolor='none')
ax.axvline(10, color='navy', linestyle='--', lw=1.5,
           label='Drop threshold (10%)')
ax.set_xlabel('Missing %', fontsize=11)
ax.set_title('Top 25 Features by Missing Data %', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## Section 5 — Data Cleaning & Preprocessing


In [ ]:
# Filter out sparse columns
sparsity_rate  = patient_records.isnull().mean()
retained_cols  = sparsity_rate[sparsity_rate < 0.10].index
working_df     = patient_records[retained_cols].copy()

print(f"Columns kept  : {working_df.shape[1]}  (from {patient_records.shape[1]})")
print(f"Columns dropped: {patient_records.shape[1] - working_df.shape[1]}")

In [ ]:
# Numeric imputation with mean
numeric_field_list = working_df.select_dtypes(include=np.number).columns
col_averages       = working_df[numeric_field_list].mean()
working_df[numeric_field_list] = working_df[numeric_field_list].fillna(col_averages)

# Categorical imputation with mode
string_field_list = working_df.select_dtypes(include='object').columns
for field in string_field_list:
    if working_df[field].isnull().any():
        dominant_val = working_df[field].mode()[0]
        working_df[field] = working_df[field].fillna(dominant_val)

leftover_nulls = working_df.isnull().sum().sum()
print(f"Nulls remaining after imputation: {leftover_nulls}")

##  Section 6 — Categorical Encoding

In [ ]:
# Identify categoricals (excluding IDs and target)
protected_cols    = ['encounter_id', 'patient_id', 'hospital_id', 'hospital_death']
string_cols       = working_df.select_dtypes(include='object').columns.tolist()
string_cols       = [s for s in string_cols if s not in protected_cols]

# Binary → Label Encoding
binary_str_cols   = [s for s in string_cols if working_df[s].nunique() == 2]
multicat_cols     = [s for s in string_cols if working_df[s].nunique() >  2]

label_enc_map = {}
for field in binary_str_cols:
    le_obj                  = LabelEncoder()
    working_df[field]       = le_obj.fit_transform(working_df[field])
    label_enc_map[field]    = le_obj

# Multi-category → One-Hot Encoding
if multicat_cols:
    working_df = pd.get_dummies(working_df, columns=multicat_cols, drop_first=True)

print(f"Label-encoded     : {len(binary_str_cols)} columns")
print(f"One-hot-encoded   : {len(multicat_cols)} columns")
print(f"Final shape       : {working_df.shape}")

## Section 7 — Train/Test Split & Imbalance Ratio

In [ ]:
# Build feature matrix (keep encounter_id for group splitting)
drop_from_X   = ['patient_id', 'hospital_id', 'hospital_death']
outcome_label = 'hospital_death'

X_all   = working_df.drop(columns=drop_from_X)
y_all   = working_df[outcome_label]

# Stratified split
X_train_raw, X_holdout_raw, y_tr, y_ho = train_test_split(
    X_all, y_all,
    test_size   = 0.25,
    stratify    = y_all,
    random_state= 77
)

# Preserve group IDs before dropping
group_ids      = X_train_raw['encounter_id'].values
X_tr           = X_train_raw.drop(columns=['encounter_id'])
X_ho           = X_holdout_raw.drop(columns=['encounter_id'])

print(f"Train: {X_tr.shape}  |  Holdout: {X_ho.shape}")

In [ ]:
# Class imbalance weight for XGBoost
n_neg, n_pos = np.bincount(y_tr)
pos_wt       = n_neg / n_pos
print(f"Class 0 (survived) in train: {n_neg:,}")
print(f"Class 1 (died)     in train: {n_pos:,}")
print(f"scale_pos_weight           : {pos_wt:.2f}")

## Section 8 — Correlation-Based Feature Pruning

In [ ]:
def prune_redundant_features(df_in, corr_threshold=0.9):
    """
    Removes one column from each highly-correlated pair.
    Only the upper triangle of the correlation matrix is scanned
    to avoid counting the same pair twice.
    """
    corr_abs   = df_in.corr().abs()
    mask_upper = np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
    upper_half = corr_abs.where(mask_upper)
    cols_drop  = [c for c in upper_half.columns
                  if any(upper_half[c] > corr_threshold)]
    return df_in.drop(columns=cols_drop), cols_drop

## Section 9 — GroupKFold Cross-Validation with Early Stopping

In [ ]:
cv_engine   = GroupKFold(n_splits=5)
auc_by_fold = []

for fold_idx, (idx_tr, idx_val) in enumerate(
        cv_engine.split(X_tr, y_tr, group_ids)):

    print(f"\n── Fold {fold_idx + 1} / 5 ──")

    # Split
    Xt = X_tr.iloc[idx_tr];   yt = y_tr.iloc[idx_tr]
    Xv = X_tr.iloc[idx_val];  yv = y_tr.iloc[idx_val]

    # Correlation filter (only on this fold's training data)
    Xt_pruned, pruned_list = prune_redundant_features(Xt, corr_threshold=0.9)
    Xv_pruned = Xv.drop(columns=pruned_list, errors='ignore')

    # ── NEW FEATURE: Early Stopping ─────────────────────────────────────────
    # Use a small internal eval set within the fold to stop when val loss
    # stops improving — avoids overfitting without a fixed n_estimators cap.
    Xt_core, Xt_es, yt_core, yt_es = train_test_split(
        Xt_pruned, yt, test_size=0.1, random_state=77
    )

    cv_clf = XGBClassifier(
        n_estimators         = 500,    # upper ceiling — early stopping will cut this
        max_depth            = 6,
        learning_rate        = 0.05,
        subsample            = 0.8,
        colsample_bytree     = 0.8,
        scale_pos_weight     = pos_wt,
        random_state         = 77,
        eval_metric          = 'logloss',
        early_stopping_rounds= 30,
        n_jobs               = -1
    )
    cv_clf.fit(
        Xt_core, yt_core,
        eval_set        = [(Xt_es, yt_es)],
        verbose         = False
    )

    val_prob     = cv_clf.predict_proba(Xv_pruned)[:, 1]
    fold_auc_val = roc_auc_score(yv, val_prob)
    auc_by_fold.append(fold_auc_val)
    best_iter    = cv_clf.best_iteration
    print(f"   Validation AUC: {fold_auc_val:.4f}  |  Best iteration: {best_iter}")

mean_cv_auc = np.mean(auc_by_fold)
std_cv_auc  = np.std(auc_by_fold)
print(f"\n Mean CV AUC: {mean_cv_auc:.4f} ± {std_cv_auc:.4f}")

In [ ]:
# ── NEW FEATURE: Per-Fold AUC Bar Chart ─────────────────────────────────────
fold_labels = [f'Fold {i+1}' for i in range(5)]
bar_colors  = ['#5B9BD5' if v >= mean_cv_auc else '#ED7D31' for v in auc_by_fold]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(fold_labels, auc_by_fold, color=bar_colors, edgecolor='white', width=0.55)
ax.axhline(mean_cv_auc, color='green', linestyle='--', lw=1.5,
           label=f'Mean AUC = {mean_cv_auc:.4f}')
ax.set_ylim(min(auc_by_fold) - 0.02, max(auc_by_fold) + 0.02)
ax.set_ylabel('AUC-ROC', fontsize=11)
ax.set_title('Per-Fold Validation AUC — GroupKFold (5 splits)', fontsize=13, fontweight='bold')
ax.legend()

for bar, val in zip(bars, auc_by_fold):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
print("Blue = above mean  |  Orange = below mean")

## Section 10 — Train Final Model

In [ ]:
# Prune correlations on full training set
X_tr_clean, final_pruned_cols = prune_redundant_features(X_tr, corr_threshold=0.9)
X_ho_clean = X_ho.drop(columns=final_pruned_cols, errors='ignore')

print(f"Features before pruning: {X_tr.shape[1]}")
print(f"Features after  pruning: {X_tr_clean.shape[1]}")

# Internal eval split for early stopping on final model
X_core, X_es_val, y_core, y_es_val = train_test_split(
    X_tr_clean, y_tr, test_size=0.1, random_state=77
)

final_clf = XGBClassifier(
    n_estimators          = 600,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    scale_pos_weight      = pos_wt,
    random_state          = 77,
    eval_metric           = 'logloss',
    early_stopping_rounds = 30,
    n_jobs                = -1
)

final_clf.fit(
    X_core, y_core,
    eval_set   = [(X_es_val, y_es_val)],
    verbose    = False
)

print(f"\n Final model trained. Best iteration: {final_clf.best_iteration}")

## Section 11 — Evaluation on Holdout Set

In [ ]:
# Predict on holdout
holdout_proba  = final_clf.predict_proba(X_ho_clean)[:, 1]
holdout_preds  = final_clf.predict(X_ho_clean)

holdout_auc    = roc_auc_score(y_ho, holdout_proba)
print(f"Holdout AUC-ROC : {holdout_auc:.4f}")
print()
print(classification_report(y_ho, holdout_preds,
                             target_names=['Survived', 'Died']))

In [ ]:
# ROC Curve plot
fpr_h, tpr_h, _ = roc_curve(y_ho, holdout_proba)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_h, tpr_h, lw=2.5, color='#5B9BD5',
        label=f'XGBoost AUC = {holdout_auc:.4f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — Holdout Set', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Section 12 — Feature Importance Bar Chart (XGBoost Built-in)

In [ ]:
# ── Feature Importance Bar Chart ────────────────────────────────
feature_importances = pd.Series(
    final_clf.feature_importances_,
    index=X_tr_clean.columns
).sort_values(ascending=False)

top_n        = 25
top_features = feature_importances.head(top_n).sort_values()

# Colour-code by importance quartile
q75 = top_features.quantile(0.75)
q50 = top_features.quantile(0.50)
bar_clrs = ['#D94F3D' if v >= q75 else
            '#F4A261' if v >= q50 else
            '#57A0D2'
            for v in top_features.values]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_features.index, top_features.values,
        color=bar_clrs, edgecolor='none', height=0.7)
ax.set_xlabel('XGBoost Feature Importance (F-score)', fontsize=11)
ax.set_title(f'Top {top_n} Predictive Features — Final Model',
             fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor='#D94F3D', label='Top quartile'),
    Patch(facecolor='#F4A261', label='2nd quartile'),
    Patch(facecolor='#57A0D2', label='Lower half')
]
ax.legend(handles=legend_els, loc='lower right')
plt.tight_layout()
plt.show()

## Section 13 — KDE of Predicted Probability Distributions


In [ ]:
# ── KDE of Predicted Probability Distributions ─────────────────
prob_survived = holdout_proba[y_ho.values == 0]
prob_died     = holdout_proba[y_ho.values == 1]

fig, ax = plt.subplots(figsize=(10, 5))

sns.kdeplot(prob_survived, ax=ax, fill=True, alpha=0.45,
            color='#4C9BE8', label=f'Survived  (n={len(prob_survived):,})')
sns.kdeplot(prob_died, ax=ax, fill=True, alpha=0.45,
            color='#E85757', label=f'Died      (n={len(prob_died):,})')

ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold = 0.5')
ax.set_xlabel('Predicted Probability of Death', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Predicted Score Distribution by Actual Outcome',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMedian predicted probability:")
print(f"  Survived patients : {np.median(prob_survived):.4f}")
print(f"  Died patients     : {np.median(prob_died):.4f}")

## Section 14 — Final Summary Dashboard

In [ ]:
print("=" * 58)
print("     HOSPITAL PATIENT SURVIVAL — FINAL REPORT")
print("=" * 58)
print(f"  Records loaded          : {len(patient_records):,}")
print(f"  Features after pruning  : {X_tr_clean.shape[1]}")
print(f"  Class imbalance ratio   : {pos_wt:.1f}:1")
print(f"  CV AUC (5-fold GroupKF) : {mean_cv_auc:.4f} ± {std_cv_auc:.4f}")
print(f"  Holdout AUC-ROC         : {holdout_auc:.4f}")
print(f"  Best model iteration    : {final_clf.best_iteration}")
print("=" * 58)
print()
print("Design decisions:")
print("  • Data sourced via Kaggle API (no manual download)")
print("  • Mean imputation for numeric features")
print("  • GroupKFold prevents encounter-level leakage")
print("  • scale_pos_weight handles class imbalance")
print("  • Early stopping prevents overfitting")
print("  • Per-fold AUC chart shows stability")
print("  • Feature importance bar chart (colour-coded quartiles)")
print("  • KDE score distribution shows model separation quality")